In [1]:
## Libraries
import pandas as pd
import numpy as np
import yfinance as yf
import sqlite3
from pathlib import Path
import matplotlib.pyplot as plt
import os

## Display Options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200) 

In [2]:
## Project Parameters (Portfolio Construction, Start/End Date, Initial Portfolio Value, Defining the Risk-Free Rate)
TICKERS = ["SMH", "VO", "IJR", "BUFR", "UGL", "SPY"] ## Example Portfolio, you may adjust if you'd like
BENCHMARK = "SPY" ## Ensure that the Benchmark is within the Portfolio listed above

TARGET_WEIGHTS = {
    "SMH": 0.15, 
    "VO": 0.20,
    "IJR": 0.15,
    "BUFR": 0.15,
    "UGL": 0.20,
    "SPY": 0.15
}

## Note. till current date, and Risk-Free Rate varies each year
START_DATE = "2021-01-01"
END_DATE = None           
INITIAL_PORTFOLIO_VALUE = 10000
RISK_FREE_RATE = 0.0353 
DB_PATH = "stock_analysis.db"

In [3]:
## Download Raw Market Data
def download_price_data(tickers, start_date, end_date):
    data = yf.download(
        tickers=tickers,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=False
    )
    adj_close = data["Adj Close"].copy()
    adj_close = adj_close.sort_index() 
    adj_close.index.name = "Date"
    return adj_close
    
prices_wide = download_price_data(TICKERS, START_DATE, END_DATE)

## Outputting the Portfolio Table
prices_wide.head()

Ticker,BUFR,IJR,SMH,SPY,UGL,VO
Date,,,,,,
2021-01-04,21.280001,83.934723,106.176178,343.319153,17.842501,47.057434
2021-01-05,21.445000,85.723335,108.034248,345.683655,17.932501,47.505512
2021-01-06,21.660000,89.890549,107.704346,347.750397,17.299999,48.295422
2021-01-07,21.660000,90.840172,112.148163,352.917084,17.209999,48.988327
2021-01-08,21.650000,90.093376,111.745499,354.927856,15.997500,49.210060


In [4]:
## Validating and Cleaning the Price Data
prices_wide.info()
prices_wide.isna().sum()

## Cleaning the Data for missing numbers
prices_wide = prices_wide.dropna(how="all")
prices_wide = prices_wide.ffill()

<class 'pandas.DataFrame'>
DatetimeIndex: 1337 entries, 2021-01-04 to 2026-04-30
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   BUFR    1337 non-null   float64
 1   IJR     1337 non-null   float64
 2   SMH     1337 non-null   float64
 3   SPY     1337 non-null   float64
 4   UGL     1337 non-null   float64
 5   VO      1337 non-null   float64
dtypes: float64(6)
memory usage: 73.1 KB


In [5]:
## Converting Prices to Long Format
def reshape_prices_long(prices_wide_df):
    prices_long = (
        prices_wide_df
        .reset_index()
        .melt(id_vars="Date", var_name="Ticker", value_name="Adj_Close")
        .sort_values(["Ticker", "Date"])
        .reset_index(drop=True)
    )
    return prices_long

## Prices in a Long Format
prices_long = reshape_prices_long(prices_wide)
prices_long.head()

,Date,Ticker,Adj_Close
0,2021-01-04,BUFR,21.280001
1,2021-01-05,BUFR,21.445000
2,2021-01-06,BUFR,21.660000
3,2021-01-07,BUFR,21.660000
4,2021-01-08,BUFR,21.650000
